# Reference Data Consolidation - App

**How to use**

1. Run the cell below (**Shift + Enter**, or *Run > Run All Cells*). A form appears.
2. Choose the **Reference Data Hierarchy**, **HC Current Month** and **Appian** files, and the **consolidation notebook** (`reference_data_consolidation.ipynb`).
3. Click **Run**. Outputs and the run log are saved to the `OUTPUT` folder next to this notebook (or the folder shown in *Save to*).

Everything runs in this Jupyter session on your PC - nothing is uploaded to the internet.
If the form shows as text instead of buttons, install ipywidgets once: open *Anaconda Prompt* and run `pip install ipywidgets`, then restart Jupyter.

In [ ]:
# ============================================================
# CONSOLIDATION APP - run this cell (Shift+Enter) to show the form
# ============================================================

import contextlib, io, json, os, re, shutil, tempfile, traceback
from datetime import datetime

import ipywidgets as w
from IPython.display import display, FileLink

RED, BLACK, GREY, LIGHT = "#db0011", "#000000", "#767676", "#f3f3f3"
DEFAULT_OUTPUT_DIR = os.path.join(os.getcwd(), "OUTPUT")
INPUT_VARS = {"reference": "reference_file", "hc": "hc_file", "appian": "appian_file"}


# ---------------------------------------------------------- running the notebook

def notebook_to_code(raw_bytes):
    # Join the code cells; Jupyter-only lines (%magic, !shell) are commented out
    nb = json.loads(bytes(raw_bytes).decode("utf-8"))
    cells = []
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)
        cells.append("\n".join(
            "# " + line if line.lstrip().startswith(("%", "!")) else line
            for line in source.splitlines()
        ))
    if not cells:
        raise ValueError("The notebook has no code cells.")
    return "\n\n".join(cells) + "\n"


def set_assignment(code, name, value):
    # Replace the line "name = ..." in section 1 of the notebook
    pattern = re.compile(rf"^{re.escape(name)} = .*$", re.MULTILINE)
    if not pattern.search(code):
        raise ValueError(
            f"Could not find the line '{name} = ...' in the notebook. "
            "Is this the consolidation notebook?"
        )
    return pattern.sub(lambda _: f"{name} = {value}", code, count=1)


def run_consolidation(inputs, notebook, output_dir, reset_hiring_flg, drop_removed):
    """inputs: {"reference"/"hc"/"appian": (name, bytes)}, notebook: (name, bytes)"""
    code = notebook_to_code(notebook[1])
    os.makedirs(output_dir, exist_ok=True)
    work_dir = tempfile.mkdtemp(prefix="consolidation_")
    try:
        for field, (name, data) in inputs.items():
            path = os.path.join(work_dir, f"{field}_{os.path.basename(name)}")
            with open(path, "wb") as f:
                f.write(bytes(data))
            code = set_assignment(code, INPUT_VARS[field], repr(path))
        code = set_assignment(code, "output_folder", repr(output_dir))
        code = set_assignment(code, "RESET_HIRING_FLG", str(bool(reset_hiring_flg)))
        code = set_assignment(code, "DROP_REMOVED_POSITIONS", str(bool(drop_removed)))

        before = set(os.listdir(output_dir))
        log, ok = io.StringIO(), True
        with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
            try:
                exec(compile(code, notebook[0], "exec"), {"__name__": "__main__"})
            except Exception:
                ok = False
                traceback.print_exc()
        created = sorted(set(os.listdir(output_dir)) - before)
        return ok, log.getvalue(), created
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)


# ---------------------------------------------------------- the form

def uploaded(widget):
    # Works with ipywidgets 7 (dict) and 8 (tuple)
    value = widget.value
    if not value:
        return None
    if isinstance(value, dict):
        name, item = next(iter(value.items()))
        return name, item["content"]
    item = value[0]
    return item["name"], item["content"]


def file_box(label, accept):
    up = w.FileUpload(accept=accept, multiple=False, description="Choose file",
                      layout=w.Layout(width="150px"))
    up.style.button_color = BLACK
    name = w.HTML(f"<span style='color:{GREY}'>No file chosen</span>")
    def changed(_):
        got = uploaded(up)
        name.value = (f"<b style='color:{BLACK}'><span style='color:{RED}'>&#10003;</span> {got[0]}</b>"
                      if got else f"<span style='color:{GREY}'>No file chosen</span>")
    up.observe(changed, names="value")
    box = w.VBox([w.HTML(f"<b style='color:{BLACK}'>{label}</b>"), up, name],
                 layout=w.Layout(border=f"1px solid #d7d8d6", padding="10px", width="230px",
                                 margin="0 10px 10px 0"))
    return up, box


def step(n, text):
    return w.HTML(f"<h3 style='margin:14px 0 8px;color:{BLACK}'>"
                  f"<span style='background:{RED};color:#fff;padding:1px 8px;margin-right:8px'>{n}</span>{text}</h3>")


ref_up, ref_box = file_box("Reference Data Hierarchy", ".xlsx,.xlsm")
hc_up, hc_box = file_box("HC Current Month", ".xlsx,.xlsm")
app_up, app_box = file_box("Appian", ".xlsx,.xlsm")
nb_up, nb_box = file_box("Consolidation notebook", ".ipynb")

reset_cb = w.Checkbox(description="Reset HIRING_FLG every run", indent=False)
drop_cb = w.Checkbox(description="Drop removed positions from the Reference output", indent=False)
out_txt = w.Text(value=DEFAULT_OUTPUT_DIR, description="Save to:", layout=w.Layout(width="90%"))

run_btn = w.Button(description="Run", layout=w.Layout(width="140px", height="40px"))
run_btn.style.button_color = RED
run_btn.style.text_color = "white"
run_btn.style.font_weight = "bold"
status = w.HTML()
results = w.Output()

header = w.HTML(
    f"<div style='background:{BLACK};color:#fff;padding:4px 12px;font-size:12px'>"
    f"Internal tool &middot; runs in this Jupyter session only</div>"
    f"<div style='border-bottom:4px solid {RED};padding:12px 12px 10px'>"
    f"<div style='font-size:20px;font-weight:bold;color:{BLACK}'>"
    f"Reference Data / Headcount / Appian Consolidation</div>"
    f"<div style='color:{GREY};font-size:13px'>Choose the three input files and the "
    f"consolidation notebook, then click Run.</div></div>")


def on_run(_):
    picked = {"reference": uploaded(ref_up), "hc": uploaded(hc_up), "appian": uploaded(app_up)}
    notebook = uploaded(nb_up)
    missing = [k for k, v in {**picked, "notebook": notebook}.items() if v is None]
    results.clear_output()
    if missing:
        status.value = f"<b style='color:{RED}'>Please choose: {', '.join(missing)}</b>"
        return

    run_btn.disabled = True
    status.value = f"<span style='color:{GREY}'>Running... please wait.</span>"
    started = datetime.now()
    try:
        ok, log, created = run_consolidation(
            picked, notebook, out_txt.value.strip() or DEFAULT_OUTPUT_DIR,
            reset_cb.value, drop_cb.value)
    except Exception as err:
        ok, log, created = False, f"{type(err).__name__}: {err}", []
    finally:
        run_btn.disabled = False

    secs = int((datetime.now() - started).total_seconds())
    status.value = (f"<b style='color:{BLACK}'><span style='color:{RED}'>&#10003;</span> "
                    f"Completed in {secs}s - {len(created)} file(s) saved</b>" if ok
                    else f"<b style='color:{RED}'>Run failed - see the log below.</b>")
    with results:
        folder = out_txt.value.strip() or DEFAULT_OUTPUT_DIR
        if created:
            display(w.HTML(f"<b>Saved to:</b> {folder}"))
            for name in created:
                path = os.path.join(folder, name)
                try:
                    display(FileLink(os.path.relpath(path)))
                except ValueError:  # different drive - show the full path instead
                    display(w.HTML(path))
        display(w.HTML(f"<b style='color:{BLACK}'>Run log</b>"))
        print(log)


run_btn.on_click(on_run)

display(w.VBox([
    header,
    step(1, "Choose input files and notebook"),
    w.HBox([ref_box, hc_box, app_box, nb_box], layout=w.Layout(flex_flow="row wrap")),
    step(2, "Options"),
    reset_cb, drop_cb, out_txt,
    step(3, "Run"),
    w.HBox([run_btn, status], layout=w.Layout(align_items="center", grid_gap="16px")),
    results,
], layout=w.Layout(border=f"1px solid #d7d8d6", padding="0 0 12px 0")))